In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

Initial Cleaning 

In [ ]:
import pandas as pd
from pathlib import Path


source_folder = Path(r'___') 
output_folder = Path("cleaned") 

output_folder.mkdir(parents=True, exist_ok=True)


all_files = list(source_folder.glob("*.csv"))

if len(all_files) == 0:
    print(f"⚠️ No CSV files found in {source_folder}. Double-check your path!")
else:
    print(f"Found {len(all_files)} files. Starting the cleaning process...\n")

    for filepath in all_files:
        try:
            df = pd.read_csv(filepath)
            
            # --- START CLEANING ---
            
            df.drop_duplicates(inplace=True)
            df.columns = [str(col).strip().lower().replace(" ", "_") for col in df.columns]
            df.fillna('NaN', inplace=True)

        
            new_filename = f"cleaned_{filepath.name}"
            
    
            save_path = output_folder / new_filename
            
    
            df.to_csv(save_path, index=False)
            
            print(f" Successfully saved: {new_filename}")
            
        except Exception as e:
        
            print(f" Error processing {filepath.name}: {e}")

    print(f"\nAll finished! Your files are safely stored locally in: {output_folder.absolute()}")

Found 9 files. Starting the cleaning process...

✅ Successfully saved: cleaned_clinical_trials.csv
✅ Successfully saved: cleaned_wikipedia_summaries.csv
✅ Successfully saved: cleaned_stock_prices.csv
✅ Successfully saved: cleaned_search_trends.csv
✅ Successfully saved: cleaned_Healthcare-Diabetes.csv
✅ Successfully saved: cleaned_data_dictionary.csv
✅ Successfully saved: cleaned_adverse_events.csv
✅ Successfully saved: cleaned_adverse_events_summary.csv
✅ Successfully saved: cleaned_drugs_overview.csv

All finished! Your files are safely stored locally in: /Users/emilyterzich/Desktop/Data/cleaned


Data Cataloging

In [ ]:
import pandas as pd


def normalize_column(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()
    return df

from pathlib import Path

cleaned_folder = Path("/Users/emilyterzich/Desktop/Data/cleaned")

drugs = normalize_column(pd.read_csv(cleaned_folder / "cleaned_drugs_overview.csv"), "generic_name")
events = normalize_column(pd.read_csv(cleaned_folder / "cleaned_adverse_events.csv"), "generic_name")

In [ ]:

brand_map = drugs[['generic_name', 'brand_names']].copy()

brand_map['brand_names'] = brand_map['brand_names'].str.split(';')
brand_map = brand_map.explode('brand_names')

brand_map = normalize_column(brand_map, 'brand_names')

In [ ]:
ticker_to_manufacturer = {
    'LLY': 'Eli Lilly',
    'NVO': 'Novo Nordisk'
}


stocks = pd.read_csv(cleaned_folder / "cleaned_stock_prices.csv")
stocks['manufacturer'] = stocks['ticker'].map(ticker_to_manufacturer)

In [ ]:

master_drug_index = pd.concat([
    brand_map.rename(columns={'brand_names': 'term', 'generic_name': 'canonical_name'}),
    drugs[['generic_name']].rename(columns={'generic_name': 'term'}).assign(canonical_name=drugs['generic_name'])
]).drop_duplicates()


In [ ]:
def search_catalog(keyword):
    keyword = keyword.lower().strip()
    
    match = master_drug_index[master_drug_index['term'] == keyword]
    if match.empty:
        return "No results found."
    
    canonical = match.iloc[0]['canonical_name']

    results = {
        "Safety Events": events[events['generic_name'] == canonical],
        "Clinical Trials": trials[trials['conditions'].str.contains(canonical, case=False)],
        "Market Performance": stocks[stocks['manufacturer'].isin(
            drugs[drugs['generic_name'] == canonical]['manufacturer']
        )]
    }
    return results

APP

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import os

app = FastAPI(title="Pharma Data Catalog API (Pandas Edition)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

master_index_data = []

def create_sample_csvs():
    """Helper: Creates sample CSVs in your folder if they don't exist."""
    if not os.path.exists("drugs_overview.csv"):
        pd.DataFrame({
            "generic_name": ["Semaglutide", "Tirzepatide"],
            "brand_names": ["Ozempic; Wegovy", "Mounjaro; Zepbound"],
            "manufacturer": ["Novo Nordisk", "Eli Lilly"],
            "indication": ["Type 2 Diabetes, Weight Loss", "Type 2 Diabetes"]
        }).to_csv("drugs_overview.csv", index=False)
        
    if not os.path.exists("stock_prices.csv"):
        pd.DataFrame({
            "ticker": ["NVO", "LLY"],
            "manufacturer": ["Novo Nordisk", "Eli Lilly"],
            "recent_close": [124.50, 752.10]
        }).to_csv("stock_prices.csv", index=False)

@app.on_event("startup")
def build_master_index():
    """This runs automatically when you start uvicorn. It builds the index using Pandas."""
    global master_index_data
    create_sample_csvs()
    
    print("Loading CSV files into Pandas...")
    drugs_df = pd.read_csv("drugs_overview.csv")
    stocks_df = pd.read_csv("stock_prices.csv")
    
    merged_df = pd.merge(drugs_df, stocks_df, on="manufacturer", how="left")
    
    print("Building search index...")
    for _, row in merged_df.iterrows():
    
        generic = str(row['generic_name'])
        brands = [b.strip() for b in str(row['brand_names']).split(';')]
        manufacturer = str(row['manufacturer'])
        ticker = str(row['ticker'])
        

        match_terms = [generic.lower(), manufacturer.lower(), ticker.lower()] + [b.lower() for b in brands]
        
    
        master_index_data.append({
            "id": generic,
            "matchTerms": match_terms,
            "data": {
                "generic": generic,
                "brands": ", ".join(brands),
                "manufacturer": manufacturer,
                "indications": str(row['indication']),
                "ticker": ticker,
                "stockPrice": f"{row['recent_close']} USD",
                "trials": "Data pending...", 
                "phase": "Data pending...",
                "reports": "Data pending...", 
                "seriousRate": "Data pending..."
            }
        })
    print("Catalog Index successfully built and ready for search!")

@app.get("/api/search")
def search_catalog(query: str):
    clean_query = query.lower().strip()
    
    for entity in master_index_data:
        if any(clean_query == term or clean_query in term for term in entity["matchTerms"]):
            return {"status": "success", "result": entity["data"]}
            
    raise HTTPException(status_code=404, detail="No results found in the current catalog index.")